# Cookbook Experiment 4: Table Pack and Artifact Validation

## Purpose

This notebook is the formal reporting layer. It verifies that all training notebooks generated required artifacts and builds compact comparison tables directly from saved histories.

## Why this matters for evaluation

Judging often requires both conceptual explanation and auditable outputs. This notebook provides a single place to verify:

- Histories exist for all three experiments
- Core plots and model files were generated
- Final comparison table summarizes the proof across configurations

In [ ]:
import os
import json
from IPython.display import Image, display

experiments = [
    ('Simple CNN', 'outputs/01_simple_cnn'),
    ('ResNet No Skip', 'outputs/02_resnet_no_skip'),
    ('ResNet With Skip', 'outputs/03_resnet_with_skip'),
]

required_files = [
    'training_history.json',
    'plots/accuracy_curves.png',
    'plots/epoch_gradient_norms.png',
    'plots/weight_deltas.png',
    'plots/table1_architecture.png',
    'plots/table2_performance.png',
    'plots/table3_training_metrics.png',
    'plots/table4_gradient_summary.png',
]

In [ ]:
histories = {}

print('Artifact validation:')
for name, out_dir in experiments:
    missing = []
    for rel in required_files:
        p = os.path.join(out_dir, rel)
        if not os.path.exists(p):
            missing.append(rel)
    status = 'OK' if not missing else 'MISSING'
    print(f'- {name}: {status}')
    if missing:
        for item in missing:
            print('   missing ->', item)

    hist_path = os.path.join(out_dir, 'training_history.json')
    if os.path.exists(hist_path):
        with open(hist_path, 'r', encoding='utf-8') as f:
            histories[name] = json.load(f)

print('\nFinal summary table:')
print('Experiment | BestTest | FinalTrain | FinalTest | GradFirst | GradLast | DeltaFirst | DeltaLast')
for name, h in histories.items():
    first = h['tracked_layers'][0]
    last = h['tracked_layers'][-1]
    print(
        f"{name} | {max(h['test_acc']):.3f} | {h['train_acc'][-1]:.3f} | {h['test_acc'][-1]:.3f} | "
        f"{h['epoch_grad_norms'][first][-1]:.6f} | {h['epoch_grad_norms'][last][-1]:.6f} | "
        f"{h['weight_deltas'][first][-1]:.6f} | {h['weight_deltas'][last][-1]:.6f}"
    )

In [ ]:
for name, out_dir in experiments:
    print('\n===', name, '===')
    for img in [
        'plots/table1_architecture.png',
        'plots/table2_performance.png',
        'plots/table3_training_metrics.png',
        'plots/table4_gradient_summary.png',
    ]:
        p = os.path.join(out_dir, img)
        if os.path.exists(p):
            print('Showing:', p)
            display(Image(filename=p))